# Rate, Leakage, and Distortion for Stationary Gaussian Sources — reproduction

This notebook reruns every numerical check quoted in the manuscript (`tit-stationary.tex`), prints the tables of Section IX, shows the recorded distortion sweep, and draws Fig. 1. It needs only `numpy` and `matplotlib`. All random instances come from one fixed seed, so the numbers are deterministic up to floating-point differences between BLAS builds.

No proof in the paper depends on these numbers; they check the statements independently of the derivations.

In [ ]:
import json, sys, platform
import numpy as np
import spectral_core as sc
import verify_spectral as vs
print('python', sys.version.split()[0], '| numpy', np.__version__, '|', platform.platform())
print('seed', vs.SEED)

## 1. Run all checks

Each line is one statement of the paper. `PASS` means the stated tolerance was met. This cell takes a few minutes: it solves window problems of dimension up to 96 with a solver that assumes no structure.

In [ ]:
status = vs.main()
out = vs.OUT
assert status == 0, out['_meta']['failed']

## 2. Theorem 3 — the period-$n$ problem splits over Fourier frequencies

The full real $pn$-dimensional program, solved with no structure assumed, against the per-frequency Hermitian programs with one common level.

In [ ]:
r = out['thm1_decoupling']
for k in ('cases', 'cost', 'X', 'kkt', 'offdiag'):
    print(f'{k:>8}: {r[k]:.3g}')

## 3. Theorem 4 — convergence to the spectral region (Table I, left)

In [ ]:
print(' alpha     n   |V_n - V| (bits)      level nu')
for row in out['thm2_limit']['table']:
    print(f"{row['alpha']:6.1f} {row['n']:5d}   {row['err']:.3e}        {row['nu']:.8f}")

## 4. Theorem 5 and Corollary 6 — the scalar pair, and disjoint bands (Fig. 1)

In [ ]:
print({k: out['thm3_scalar_closed_form'][k] for k in ('val', 'alloc', 'thr')})
b = out['cor_disjoint_bands']
print({k: b[k] for k in b if k not in ('ok', 'seconds')})
print('leakage factor', round(b['L_of_rate_opt'] / b['L_min'], 2), '| rate factor', round(b['R_of_leak_opt'] / b['R_min'], 2))

In [ ]:
%matplotlib inline
import plot_bands
fig = plot_bands.main()

### Distortion sweep (Table II)

The sweep takes about twelve minutes, so this cell shows the recorded `sweep_bands.json`. Set `RERUN_SWEEP = True` to recompute it with `sweep_bands.py`.

In [ ]:
RERUN_SWEEP = False
import sweep_bands
sw = sweep_bands.main() if RERUN_SWEEP else json.load(open('sweep_bands.json'))
print(' D/Dmax  rate  leak  both    R_min    L_min   L-factor R-factor')
for r in sw['rows']:
    print(f"{r['frac']:7.2f} {r['bands_rate']:5d} {r['bands_leak']:5d} {r['overlap']:5d}   {r['R_min']:.4f}   {r['L_min']:.4f}    {r['leak_factor']:.2f}     {r['rate_factor']:.2f}")
print('supports disjoint for D/Dmax >=', round(sw['onset_hi'], 4), '| overlap at', round(sw['onset_lo'], 4))

## 5. Proposition 7 — a delay changes nothing

In [ ]:
print(out['prop_delay_invariance'])

## 6. Theorems 10 and 15 — windows of the stationary source (Table I, right)

Affine spectrum: exact splitting in the Toeplitz eigenbasis and convergence. General spectrum: the computed gap, which Theorem 15 says is nonnegative and tends to zero.

In [ ]:
print('affine spectrum:  N  alpha   window cost    gap (bits)    N*gap')
for row in out['thm4_affine_toeplitz']['table']:
    print(f"{row['n']:20d} {row['alpha']:5.1f}   {row['cost_eig']:.8f}   {row['gap']:+.3e}   {row['n_gap']:+.4f}")
print('exact-splitting error', out['thm4_affine_toeplitz']['decoupling_err'])
print()
print('general spectrum: alpha  N   window cost    gap (bits)    N*gap')
for row in out['conj1_general_toeplitz_evidence']['table']:
    print(f"{row['alpha']:22.1f} {row['n']:3d}   {row['cost']:.8f}   {row['gap']:+.3e}   {row['n_gap']:+.4f}")

## 6b. Theorem 16 — covariance steps of the achievability proof

The coding theorem is not something a computation verifies. This checks the steps of its proof that are statements about covariances: the decomposition, block independence, the coupling, and the two leakage inequalities.

In [ ]:
import verify_achievability as va
assert va.main() == 0

## 6c. Theorem 15 — steps of the proof of the window theorem

Unwhitened form, order sandwich, subadditivity, finite-memory embedding, sign of the gap, the block-Toeplitz feasible point, and both bounds on the gap. About a minute.

In [ ]:
import verify_windows as vw
assert vw.main() == 0
for r in vw.OUT['toeplitz_point_and_bound']['table']:
    print(f"alpha={r['alpha']:.1f} N={r['N']:3d}  gap={r['gap_window']:.4e}  first bound={r['bound_s']:.4e}  explicit bound={r['bound']:.3e}")

## 7. Proposition 8 — what the holder keeps

In [ ]:
r = out['prop_retention']
for k in r:
    if k not in ('ok', 'seconds'):
        print(f'{k:>22}: {r[k]:.10g}')
rho2, Dn = 0.64, 0.4
for name in ('q_slice', 'q_prefix0', 'q_prefix3', 'q_path'):
    q = r[name]
    print(name, 'leakage (bits):', round(float(sc.ell_scalar(rho2, 1 / (1 - q), Dn)), 4))
print('rate (no holder):', round(0.5 * np.log2(1 / Dn), 4))

## 8. Summary

In [ ]:
m = out['_meta']
print('failed:', m['failed'], '| seconds:', m['seconds'])
print('all checks passed' if not m['failed'] else 'FAILURES')